# human_eval_check
Analyses the blind head-to-head against `human_eval/prereg.txt`, frozen before the first comparison. Joins `responses.jsonl`, which records only A / B / tie, to `key.jsonl`, which holds the side assignment — that join is the only place the study is unblinded, and it happens after every judgement is recorded.

Pre-registered analyses are labelled explicitly; anything computed afterwards is marked post-hoc. Section order follows the analysis flow rather than the order the pre-registration lists.

Reads scores and responses only; no GPU, no API calls.

## 1. Load, integrity, pre-registration

In [ ]:
from google.colab import drive
import os, json, math, random, statistics as st
import numpy as np
from collections import defaultdict, Counter
from scipy import stats
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
HE, SC = f"{DRIVE}/human_eval", f"{DRIVE}/scores"
ARMS = ["lora_r8_step1000", "qlora_r8_step2000", "sp_iii_p139_step2000"]
DIMS = ["adherence", "consistency", "quality", "overall"]
DOMAIN_DELTA = {"lora_r8_step1000": -0.120, "qlora_r8_step2000": -0.080,
                "sp_iii_p139_step2000": -0.826}    # GPT-4.1 Overall, 488 rows, from lora/qlora/soft_prompt_check

KEY = {}
for l in open(f"{HE}/key.jsonl", encoding="utf-8"):
    r = json.loads(l); KEY[r["pair_id"]] = r

# a truncated final line is possible if a runtime died mid-write; the last valid record per
# pair_id wins, so a pair rated twice after such a truncation resolves to the later judgement
R, dropped = {}, 0
for l in open(f"{HE}/responses.jsonl", encoding="utf-8"):
    try: r = json.loads(l)
    except Exception: dropped += 1; continue
    R[r["pair_id"]] = r

rated   = {k: v for k, v in R.items() if not v.get("skipped")}
skipped = [k for k, v in R.items() if v.get("skipped")]
scored  = [p for p in KEY.values() if "repeat_of" not in p]
repeats = [p for p in KEY.values() if "repeat_of" in p]

print(f"comparisons in the list {len(KEY)} | responses {len(R)} | rated {len(rated)} | "
      f"skipped {len(skipped)} | unparseable lines {dropped}")
print(f"scored pairs {len(scored)} | repeats {len(repeats)}")
missing = [p["pair_id"] for p in KEY.values() if p["pair_id"] not in rated]
print(f"not yet rated: {len(missing)}" + (f" {missing[:5]}" if missing else ""))
assert not missing, "the eval is incomplete"
assert all(r["rater"] == next(iter(rated.values()))["rater"] for r in rated.values()), "mixed raters"
s = [r["seconds"] for r in rated.values()]
print(f"rater {next(iter(rated.values()))['rater']} | median {st.median(s):.0f}s per comparison | "
      f"{sum(s)/3600:.1f} h total")
print()
print(open(f"{HE}/prereg.txt", encoding="utf-8").read())

Mounted at /content/drive
comparisons in the list 165 | responses 165 | rated 165 | skipped 0 | unparseable lines 0
scored pairs 150 | repeats 15
not yet rated: 0
rater S | median 138s per comparison | 7.5 h total

Primary: agreement between the rater's overall preference and the sign of the GPT-4.1 Overall
margin on the same row, pooled over the 150 scored pairs, reported against the GPT margin band.
Rows where the rater answers tie, and rows where the GPT margin is exactly zero, are excluded
from the agreement rate and reported as counts.

Secondary, all pre-specified:
  per-contrast win rate over 50 pairs, ties excluded, two-sided binomial, 33/50 needed at p<0.05
  per-dimension agreement, testing whether the LoRA effect is confined to subject consistency
  error tag counts by contrast, as the qualitative taxonomy
  self-agreement over the 15 repeated pairs, as the single-rater reliability bound
  left-preference rate over all comparisons, as a position-bias check, expected near 0.5

## 2. Unblinding
The rater recorded a side. `left_is` says which run was on that side. Everything downstream reads the resolved value, never A or B.

In [ ]:
def to_run(pair_id, choice):
    """A/B/tie as recorded -> 'arm' | 'base' | 'tie'."""
    if choice == "tie": return "tie"
    left = KEY[pair_id]["left_is"]                       # 'base' or 'arm'
    return (left if choice == "A" else ("arm" if left == "base" else "base"))

for pid in list(KEY)[:3]:
    k = KEY[pid]
    print(f"  {pid}  A was the {k['left_is']:4s} | rater said {rated[pid]['overall']:4s} "
          f"-> preferred the {to_run(pid, rated[pid]['overall'])}")
assert to_run("p0000", "tie") == "tie"
_c = Counter(KEY[p]["left_is"] for p in KEY)
print(f"\nside assignment across the list: {dict(_c)}")

  p0000  A was the base | rater said B    -> preferred the arm
  p0001  A was the arm  | rater said A    -> preferred the arm
  p0002  A was the arm  | rater said B    -> preferred the base

side assignment across the list: {'base': 72, 'arm': 93}


## 3. Primary — agreement with GPT-4.1
Pre-registered: agreement between the rater's overall preference and the sign of the GPT-4.1 Overall margin on the same row, pooled over the 150 scored pairs, reported against the margin band. Rater ties and rows where the margin is exactly zero are excluded and reported as counts.

In [ ]:
BANDS = ["0-0.5", "0.5-1.5", ">1.5"]

# GPT-4.1 counterpart per rater dimension: PF for adherence, SC for consistency, Overall for
# overall preference. quality has no counterpart and stays on the Overall margin, descriptive only.
_S = {}
for _lab in ARMS + ["baseline"]:
    _d = {}
    for _l in open(f"{SC}/test__{_lab}.jsonl", encoding="utf-8"):
        _r = json.loads(_l)
        if "error" not in _r: _d[_r["uid"]] = _r
    _S[_lab] = _d
GPT_FOR_DIM = {"adherence": "PF", "consistency": "SC", "quality": None, "overall": None}

def _margin(p, dim):
    f = GPT_FOR_DIM[dim]
    if f is None: return p["gpt_margin"]
    return _S[p["contrast"]][p["uid"]][f] - _S["baseline"][p["uid"]][f]

def agreement(dim, subset=None):
    sub = subset if subset is not None else scored
    ag = tot = zero = tie = 0
    byband = defaultdict(lambda: [0, 0])
    for p in sub:
        h = to_run(p["pair_id"], rated[p["pair_id"]][dim])
        m = _margin(p, dim)
        if m == 0: zero += 1; continue
        if h == "tie": tie += 1; continue
        g = "arm" if m > 0 else "base"
        tot += 1; ag += (h == g)
        byband[p["gpt_band"]][0] += (h == g); byband[p["gpt_band"]][1] += 1
    return ag, tot, byband, tie, zero

ag, tot, byband, tie, zero = agreement("overall")
# agreement() short-circuits on the GPT-zero test, so its tie counter omits rows that are both.
# The two exclusion sets are counted independently here and the union is what leaves 63.
_z = {p["pair_id"] for p in scored if _margin(p, "overall") == 0}
_t = {p["pair_id"] for p in scored if rated[p["pair_id"]]["overall"] == "tie"}
print(f"excluded: GPT margin exactly zero {len(_z)} | rater tie {len(_t)} | both {len(_z & _t)} | unique {len(_z | _t)}")
# sqrt(0.25/n) is the null SE and is right for the test against 0.5, but not for an
# interval around the observed rate; the interval is Wilson, which is well behaved at
# this n and does not run past 0 or 1
_p, _z = ag / tot, 1.96
_c = _p + _z**2 / (2*tot)
_hw = _z * math.sqrt(_p*(1-_p)/tot + _z**2/(4*tot**2))
_lo, _hi = (_c - _hw) / (1 + _z**2/tot), (_c + _hw) / (1 + _z**2/tot)
print(f"AGREEMENT {ag}/{tot} = {ag/tot:.3f}   null SE {math.sqrt(0.25/tot):.3f}   "
      f"Wilson 95% CI [{_lo:.3f}, {_hi:.3f}]   "
      f"binomial vs 0.5 p {stats.binomtest(ag, tot, 0.5).pvalue:.4f}")
print(f"\n{'GPT margin band':16s} {'n':>4s} {'agree':>6s} {'rate':>7s} {'p':>8s}")
for b in BANDS:
    a, n = byband[b]
    if n: print(f"{b:16s} {n:4d} {a:6d} {a/n:7.3f} {stats.binomtest(a, n, 0.5).pvalue:8.3f}")
print("\nagreement is highest in the widest-margin band but is not monotone across bands:")
print("the middle band sits at chance on n=19, so this supports the judge where the margin")
print("is wide and is unresolved in between")

excluded: GPT margin exactly zero 36 | rater tie 64 | both 13 | unique 87
AGREEMENT 39/63 = 0.619   null SE 0.063   Wilson 95% CI [0.496, 0.729]   binomial vs 0.5 p 0.0769

GPT margin band     n  agree    rate        p
0-0.5              19     12   0.632    0.359
0.5-1.5            19      9   0.474    1.000
>1.5               25     18   0.720    0.043

agreement is highest in the widest-margin band but is not monotone across bands:
the middle band sits at chance on n=19, so this supports the judge where the margin
is wide and is unresolved in between


## 4. Per-dimension
Pre-registered: whether the LoRA effect is confined to subject consistency. The first table pools all three contrasts and is descriptive; the per-contrast table below it is what addresses the LoRA question. Each dimension is compared against its GPT-4.1 counterpart: PF for adherence, SC for consistency, Overall for overall preference. `quality` has no counterpart — GPT-4.1 PF/SC carries no dedicated general image-quality axis, and PF is instructed not to assess artistic quality, though SC can penalise anatomy or shape defects where they bear on subject identity — so no cross-instrument agreement rate is reported for it; human quality preferences are analysed separately below.

In [ ]:
print(f"{'dimension':12s} {'agree':>10s} {'rate':>7s} {'rater ties':>11s} {'tie rate':>9s}")
for d in DIMS:
    a, t, _, _, _ = agreement(d)
    ti = sum(1 for p in scored if rated[p["pair_id"]][d] == "tie")   # all rater ties, not only those outside the GPT-zero set
    if d == "quality":
        print(f"{d:12s} {'n/a':>10s} {'n/a':>7s} {ti:11d} {ti/len(scored):9.2f}   no direct GPT-4.1 counterpart")
    else:
        print(f"{d:12s} {f'{a}/{t}':>10s} {a/t if t else float('nan'):7.3f} {ti:11d} {ti/len(scored):9.2f}")

print("\nrater tie rate per dimension, over all 150 scored pairs:")
for d in DIMS:
    t = sum(1 for p in scored if rated[p["pair_id"]][d] == "tie")
    print(f"  {d:12s} {t:3d}/150 = {t/len(scored):.2f}")
print("a high tie rate is a result, not missing data: GPT-4.1 measured PF deltas of +0.000")
print("and +0.085 for the two adapters, and indifference on adherence says the same thing")

print("\nper-dimension, per-contrast preference (ties excluded):")
print(f"  {'contrast':22s} " + " ".join(f"{d:>16s}" for d in DIMS))
for lab in ARMS:
    sub = [p for p in scored if p["contrast"] == lab]
    cells = []
    for d in DIMS:
        w = sum(1 for p in sub if to_run(p["pair_id"], rated[p["pair_id"]][d]) == "arm")
        l = sum(1 for p in sub if to_run(p["pair_id"], rated[p["pair_id"]][d]) == "base")
        cells.append(f"{w}-{l}" + (f" ({w/(w+l):.2f})" if w+l else ""))
    print(f"  {lab:22s} " + " ".join(f"{c:>16s}" for c in cells))

dimension         agree    rate  rater ties  tie rate
adherence           7/9   0.778         136      0.91
consistency       15/21   0.714         109      0.73
quality             n/a     n/a          86      0.57   no direct GPT-4.1 counterpart
overall           39/63   0.619          64      0.43

rater tie rate per dimension, over all 150 scored pairs:
  adherence    136/150 = 0.91
  consistency  109/150 = 0.73
  quality       86/150 = 0.57
  overall       64/150 = 0.43
a high tie rate is a result, not missing data: GPT-4.1 measured PF deltas of +0.000
and +0.085 for the two adapters, and indifference on adherence says the same thing

per-dimension, per-contrast preference (ties excluded):
  contrast                      adherence      consistency          quality          overall
  lora_r8_step1000             3-1 (0.75)       1-7 (0.12)       6-6 (0.50)      8-11 (0.42)
  qlora_r8_step2000            2-2 (0.50)       3-9 (0.25)      11-3 (0.79)      14-9 (0.61)
  sp_iii_p139_ste

## 5. Per-contrast win rate
Pre-registered: 50 pairs per contrast, ties excluded, two-sided binomial.

The sample was stratified on the GPT-4.1 margin with an equal quota per band, so it deliberately over-represents ties and wide margins and the raw rate is not an estimate of the population preference. The reweighting below maps the cells back onto each arm's own band distribution over the 488 rows; it is post-hoc and not in the pre-registration.

In [ ]:
ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))
def load(name):
    d = {}
    for l in open(f"{SC}/{name}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if "error" not in r: d[r["uid"]] = r
    return d
def band(x):
    a = abs(x)
    return "tie" if a == 0 else ("0-0.5" if a <= 0.5 else ("0.5-1.5" if a <= 1.5 else ">1.5"))

B = load("test__baseline")
POP = {}
for lab in ARMS:
    A = load(f"test__{lab}")
    K = sorted(set(A) & set(B))
    POP[lab] = Counter(band(ov(A[k]) - ov(B[k])) for k in K)

print(f"{'contrast':22s} {'arm':>4s} {'base':>5s} {'tie':>4s} {'raw rate':>9s} {'p':>7s} "
      f"{'reweighted':>11s} {'GPT delta':>10s}")
for lab in ARMS:
    sub = [p for p in scored if p["contrast"] == lab]
    w = sum(1 for p in sub if to_run(p["pair_id"], rated[p["pair_id"]]["overall"]) == "arm")
    l = sum(1 for p in sub if to_run(p["pair_id"], rated[p["pair_id"]]["overall"]) == "base")
    t = len(sub) - w - l
    tot = sum(POP[lab].values()); num = den = 0.0
    for b, n in POP[lab].items():
        cell = [p for p in sub if p["gpt_band"] == b]
        cw = sum(1 for p in cell if to_run(p["pair_id"], rated[p["pair_id"]]["overall"]) == "arm")
        cl = sum(1 for p in cell if to_run(p["pair_id"], rated[p["pair_id"]]["overall"]) == "base")
        if cw + cl: num += (n/tot) * (cw/(cw+cl)); den += n/tot
    print(f"{lab:22s} {w:4d} {l:5d} {t:4d} {w/(w+l) if w+l else float('nan'):9.3f} "
          f"{stats.binomtest(w, w+l, 0.5).pvalue:7.3f} {num/den if den else float('nan'):11.3f} "
          f"{DOMAIN_DELTA[lab]:+10.3f}")
print(f"\nsample GPT mean margin against the population mean, showing what the stratification did:")
for lab in ARMS:
    sub = [p for p in scored if p["contrast"] == lab]
    print(f"  {lab:22s} sample {st.mean([p['gpt_margin'] for p in sub]):+.3f}  "
          f"population {DOMAIN_DELTA[lab]:+.3f}  bands {dict(POP[lab])}")

contrast                arm  base  tie  raw rate       p  reweighted  GPT delta
lora_r8_step1000          8    11   31     0.421   0.648       0.417     -0.120
qlora_r8_step2000        14     9   27     0.609   0.405       0.522     -0.080
sp_iii_p139_step2000      9    35    6     0.205   0.000       0.196     -0.826

sample GPT mean margin against the population mean, showing what the stratification did:
  lora_r8_step1000       sample +0.111  population -0.120  bands {'tie': 168, '0.5-1.5': 178, '0-0.5': 77, '>1.5': 65}
  qlora_r8_step2000      sample -0.045  population -0.080  bands {'0-0.5': 76, 'tie': 173, '0.5-1.5': 177, '>1.5': 62}
  sp_iii_p139_step2000   sample -0.420  population -0.826  bands {'0-0.5': 65, 'tie': 97, '0.5-1.5': 161, '>1.5': 165}


## 6. Reliability and bias
Pre-registered: self-agreement over the 15 repeated pairs, as the single-rater reliability bound, and the left-preference rate, expected near half. The pre-registration specifies the left-preference rate over all comparisons; the test below restricts it to the 150 primary pairs, a post-hoc independence correction. Both sets are reported: consistency reads 11-30 (p 0.0043) on the primary set and 16-30 (p 0.0541) over all 165. Repeats were shown with the sides swapped and are compared after resolving to run, so a rater who merely favoured a screen position would disagree with themselves.

In [ ]:
print(f"{'dimension':12s} {'self-agree':>11s} {'rate':>7s}")
for d in DIMS:
    same = sum(1 for p in repeats
               if to_run(p["pair_id"], rated[p["pair_id"]][d])
               == to_run(p["repeat_of"], rated[p["repeat_of"]][d]))
    print(f"{d:12s} {f'{same}/{len(repeats)}':>11s} {same/len(repeats):7.2f}")
gaps = [p["seq"] - KEY[p["repeat_of"]]["seq"] for p in repeats]
print(f"repeat separation: {min(gaps)} to {max(gaps)} comparisons apart, sides swapped")

# pre-registered set first: the left-preference rate over all comparisons
print(f"\n{'dimension':12s} {'A':>4s} {'B':>4s} {'left rate':>10s} {'p':>7s}   (all 165, as pre-registered)")
for d in DIMS:
    a = sum(1 for r in rated.values() if r[d] == "A")
    b = sum(1 for r in rated.values() if r[d] == "B")
    print(f"{d:12s} {a:4d} {b:4d} {a/(a+b):10.3f} {stats.binomtest(a, a+b, 0.5).pvalue:7.3f}")

# post-hoc independence correction: the 15 repeats are the same comparisons shown again with the
# sides swapped, so pooling them counts dependent observations twice
_primary_ids = {p["pair_id"] for p in scored}
print(f"\n{'dimension':12s} {'A':>4s} {'B':>4s} {'left rate':>10s} {'p':>7s}   (150 primary pairs, post-hoc)")
for d in DIMS:
    a = sum(1 for k, r in rated.items() if k in _primary_ids and r[d] == "A")
    b = sum(1 for k, r in rated.items() if k in _primary_ids and r[d] == "B")
    print(f"{d:12s} {a:4d} {b:4d} {a/(a+b):10.3f} {stats.binomtest(a, a+b, 0.5).pvalue:7.3f}")
print("a rate near half gives no detectable evidence of a position effect in that dimension;")
print("consistency departs from half and is carried into the side-stratified check below")

# side assignment was randomised per pair, not balanced within contrast, so the realised split is
# reported and each contrast is re-read within side. A conclusion that survives both assignments
# does not rest on the imbalance.
# target-block sensitivity: some rated rows share a target scene, and the project applies
# cluster-robust inference elsewhere. Exact binomial stays primary; this is the robustness check.
def _blockp(items, pick, n_perm=20000, seed=0):
    rng = random.Random(seed)
    blocks = {}
    for it, s in items: blocks.setdefault(it, []).append(s)
    obs = sum(sum(v) for v in blocks.values())
    tot = sum(len(v) for v in blocks.values())
    if tot == 0: return float("nan")
    cen = tot / 2.0; hits = 0
    for _ in range(n_perm):
        acc = 0
        for v in blocks.values():
            acc += sum(v) if rng.random() < 0.5 else len(v) - sum(v)
        if abs(acc - cen) >= abs(obs - cen): hits += 1
    return (hits + 1) / (n_perm + 1)

print("\ntarget-block sign-flip sensitivity (exact binomial remains primary):")
for con in sorted({p["contrast"] for p in scored}):
    rs = [p for p in scored if p["contrast"] == con]
    for dim in DIMS:
        items = []
        for p in rs:
            pref = to_run(p["pair_id"], rated[p["pair_id"]][dim])
            if pref in ("arm", "base"):
                items.append((p["uid"].split("|")[0], 1 if pref == "arm" else 0))
        a = sum(s for _, s in items); b = len(items) - a
        print(f"  {con:24s} {dim:12s} arm {a:3d} - base {b:3d}   block p {_blockp(items, 'arm'):.4f}")

# target-block position test over the 150 independent primary comparisons
print("\ntarget-block position sensitivity (primary pairs only):")
for d in DIMS:
    _it = [(p["uid"].split("|")[0], 1 if rated[p["pair_id"]][d] == "A" else 0)
           for p in scored if rated[p["pair_id"]][d] in ("A", "B")]
    _a = sum(s for _, s in _it); _b = len(_it) - _a
    print(f"  {d:12s} left {_a:3d} - right {_b:3d}   block p {_blockp(_it, 'A'):.4f}")

# pooled primary agreement, target-clustered. The 63 analysable decisions span 46 target
# scenes, so they are not independent. Cluster bootstrap over target blocks; exact binomial
# above remains primary.
_ag_items = []
for p in scored:
    if p["gpt_margin"] == 0: continue
    h = to_run(p["pair_id"], rated[p["pair_id"]]["overall"])
    if h == "tie": continue
    g = "arm" if p["gpt_margin"] > 0 else "base"
    _ag_items.append((p["uid"].split("|")[0], 1 if h == g else 0))
_blk = defaultdict(list)
for t, s in _ag_items: _blk[t].append(s)
_keys = sorted(_blk)
_rng, _B = random.Random(0), 20000
_rates = []
for _ in range(_B):
    _flat = [s for _t in _keys for s in _blk[_rng.choice(_keys)]]
    _rates.append(sum(_flat) / len(_flat))
_rates.sort()
_lo, _hi = _rates[int(0.025 * _B)], _rates[int(0.975 * _B) - 1]
# The bootstrap distribution is centred on the observed rate, not on the null, so its tail mass
# is not a null test. The interval is descriptive; the p-value comes from the same target-block
# sign-flip procedure used elsewhere in this cell.
print(f"\ntarget-clustered agreement: {sum(s for _, s in _ag_items)}/{len(_ag_items)} decisions "
      f"over {len(_keys)} targets, rate {sum(s for _, s in _ag_items)/len(_ag_items):.3f}")
print(f"  cluster bootstrap B={_B} seed=0: 95% CI [{_lo:.3f}, {_hi:.3f}] (descriptive interval)")
print(f"  target-block sign-flip B=20,000 seed 0: two-sided p {_blockp(_ag_items, 'agree'):.4f}")

print(f"\n{'contrast':24s} {'arm-left':>9s} {'p':>7s}")
for con in sorted({p["contrast"] for p in scored}):
    rs = [p for p in scored if p["contrast"] == con]
    al = sum(1 for p in rs if p["left_is"] == "arm")
    print(f"{con:24s} {f'{al}/{len(rs)}':>9s} {stats.binomtest(al, len(rs), 0.5).pvalue:7.4f}")

print(f"\n{'contrast':24s} {'side':>10s} {'dimension':12s} {'arm':>4s} {'base':>5s}")
for con in sorted({p["contrast"] for p in scored}):
    for side, lab in (("arm", "arm left"), ("base", "arm right")):
        rs = [p for p in scored if p["contrast"] == con and p["left_is"] == side]
        for d in ("consistency", "overall"):
            pref = [to_run(p["pair_id"], rated[p["pair_id"]][d]) for p in rs]
            print(f"{con:24s} {lab:>10s} {d:12s} "
                  f"{sum(1 for x in pref if x == 'arm'):4d} {sum(1 for x in pref if x == 'base'):5d}")

dimension     self-agree    rate
adherence          11/15    0.73
consistency        13/15    0.87
quality            11/15    0.73
overall            15/15    1.00
repeat separation: 32 to 150 comparisons apart, sides swapped

dimension       A    B  left rate       p   (all 165, as pre-registered)
adherence       7    9      0.438   0.804
consistency    16   30      0.348   0.054
quality        33   36      0.478   0.810
overall        40   53      0.430   0.213

dimension       A    B  left rate       p   (150 primary pairs, post-hoc)
adherence       6    8      0.429   0.791
consistency    11   30      0.268   0.004
quality        31   33      0.484   0.901
overall        37   49      0.430   0.235
a rate near half gives no detectable evidence of a position effect in that dimension;
consistency departs from half and is carried into the side-stratified check below

target-block sign-flip sensitivity (exact binomial remains primary):
  lora_r8_step1000         adherence    arm   3 - 

## 7. Where GPT-4.1 is blind
Post-hoc. The rows the judge scored dead level, and the tag counts, are the part of the leg that adds information rather than corroborating it: GPT-4.1 PF/SC carries no dedicated general image-quality axis. PF is instructed not to assess artistic quality or aesthetic appeal; SC does direct attention to body shape, skin tone and abnormal anatomical structures, but only where these bear on subject identity. Artifacts and coherence independent of identity are unscored.

In [ ]:
z = [p for p in scored if p["gpt_margin"] == 0]
print(f"pairs GPT-4.1 scored exactly level: {len(z)}")
for d in DIMS:
    c = Counter(to_run(p["pair_id"], rated[p["pair_id"]][d]) for p in z)
    print(f"  {d:12s} tie {c['tie']:3d}  arm {c['arm']:3d}  base {c['base']:3d}"
          f"   -> a difference seen on {len(z)-c['tie']}/{len(z)}")

print("\nerror tags by contrast:")
for lab in ARMS:
    sub = [p for p in scored if p["contrast"] == lab]
    tg = Counter(t for p in sub for t in rated[p["pair_id"]].get("tags", []))
    print(f"  {lab:22s} {dict(tg.most_common())}")
tg = Counter(t for r in rated.values() for t in r.get("tags", []))
print(f"  {'ALL':22s} {dict(tg.most_common())}")

notes = [(p["pair_id"], p["contrast"], rated[p["pair_id"]]["note"])
         for p in KEY.values() if rated[p["pair_id"]].get("note")]
print(f"\nfree-text notes: {len(notes)}")
for pid, lab, n in notes:
    print(f"  [{lab[:12]:12s}] {n[:150]}")

pairs GPT-4.1 scored exactly level: 36
  adherence    tie  35  arm   0  base   1   -> a difference seen on 1/36
  consistency  tie  24  arm   2  base  10   -> a difference seen on 12/36
  quality      tie  17  arm   8  base  11   -> a difference seen on 19/36
  overall      tie  13  arm   9  base  14   -> a difference seen on 23/36

error tags by contrast:
  lora_r8_step1000       {'artifacts or image quality': 10, 'identity / likeness wrong': 6, 'scene or action does not match the instruction': 2, 'colour or material wrong': 2, 'other': 1, 'markings, logo or text wrong': 1}
  qlora_r8_step2000      {'artifacts or image quality': 12, 'identity / likeness wrong': 9, 'colour or material wrong': 4, 'scene or action does not match the instruction': 1, 'markings, logo or text wrong': 1}
  sp_iii_p139_step2000   {'artifacts or image quality': 33, 'identity / likeness wrong': 18, 'markings, logo or text wrong': 3, 'scene or action does not match the instruction': 3, 'colour or material wrong'

## 8. Cross-instrument summary

In [ ]:
print(f"{'arm':22s} {'GPT-4.1 domain':>15s} {'human arm rate':>15s} {'human p':>9s} {'verdict':>10s}")
for lab in ARMS:
    sub = [p for p in scored if p["contrast"] == lab]
    w = sum(1 for p in sub if to_run(p["pair_id"], rated[p["pair_id"]]["overall"]) == "arm")
    l = sum(1 for p in sub if to_run(p["pair_id"], rated[p["pair_id"]]["overall"]) == "base")
    p_ = stats.binomtest(w, w+l, 0.5).pvalue
    # the human leg is powered to detect only a large preference shift at 50 pairs per
    # contrast, so a non-significant result is a null, never equivalence, and is not
    # compared against the GPT delta by any threshold
    v = ("human evaluation rejects this arm" if p_ < 0.05
        else f"no detectable human preference difference (non-tie n={w+l})")
    print(f"{lab:22s} {DOMAIN_DELTA[lab]:+15.3f} {w/(w+l):15.3f} {p_:9.3f} {v}")
ag, tot, _, _, _ = agreement("overall")
_zs = {p["pair_id"] for p in scored if _margin(p, "overall") == 0}
_ts = {p["pair_id"] for p in scored if rated[p["pair_id"]]["overall"] == "tie"}
print(f"\npooled agreement {ag}/{tot} = {ag/tot:.3f} (p {stats.binomtest(ag,tot,0.5).pvalue:.3f}); "
      f"{len(_ts)} rater ties and {len(_zs)} zero-margin rows, {len(_zs & _ts)} in both, "
      f"{len(_zs | _ts)} rows excluded")
same = sum(1 for p in repeats if to_run(p["pair_id"], rated[p["pair_id"]]["overall"])
           == to_run(p["repeat_of"], rated[p["repeat_of"]]["overall"]))
print(f"rater self-agreement on overall {same}/{len(repeats)}; the reliability bound on everything above")

arm                     GPT-4.1 domain  human arm rate   human p    verdict
lora_r8_step1000                -0.120           0.421     0.648 no detectable human preference difference (non-tie n=19)
qlora_r8_step2000               -0.080           0.609     0.405 no detectable human preference difference (non-tie n=23)
sp_iii_p139_step2000            -0.826           0.205     0.000 human evaluation rejects this arm

pooled agreement 39/63 = 0.619 (p 0.077); 64 rater ties and 36 zero-margin rows, 13 in both, 87 rows excluded
rater self-agreement on overall 15/15; the reliability bound on everything above
